In [ ]:
import sys
import os
from pathlib import Path

# Add src to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'src'))

import random
import pandas as pd
from datetime import datetime, timedelta
from data_simulation.gas_turbine import GasTurbine
from data_simulation.compressor import Compressor
from data_simulation.pump import Pump
from data_simulation.physics.environmental_conditions import EnvironmentalConditions, LocationType
from data_simulation.physics.weather_api_client import create_hybrid_environment
from data_simulation.ml_utils.ml_output_modes import OutputMode
from ingestion.equipment_sim import simulate_equipment
from ingestion.data_pipeline import DataPipeline
import sqlite3
import json
from sqlalchemy import create_engine, text

In [ ]:
# Simulation Configuration
CONFIG = {
    # Simulation time
    'start_date': datetime(2025, 7, 1),
    'duration_days': 180,
    'sample_interval_min': 15,
    
    # Equipment counts
    'n_turbines': 15,
    'n_compressors': 15,
    'n_pumps': 40,
    
    # Degradation
    'degradation_multiplier': 1.5,
    
    # Initial health ranges per equipment type
    'health_ranges': {
        'turbine': {
            'hgp': (0.70, 0.98),
            'blade': (0.70, 0.98),
            'bearing': (0.70, 0.98),
            'fuel': (0.70, 0.98),
        },
        'compressor': {
            'impeller': (0.88, 0.98),
            'bearing': (0.85, 0.98),
            'seal_primary': (0.85, 0.98),
            'seal_secondary': (0.85, 0.98),
        },
        'pump': {
            'impeller': (0.70, 0.98),
            'seal': (0.70, 0.98),
            'bearing_de': (0.70, 0.98),
            'bearing_nde': (0.70, 0.98),
        }
    }
}

print("Simulation Configuration:")
for key, value in CONFIG.items():
    if key != 'health_ranges':
        print(f"  {key}: {value}")
print(f"  Health ranges: {len(CONFIG['health_ranges'])} equipment types configured")

In [ ]:
def generate_initial_health(config: dict, equipment_type: str) -> dict:
    """Generate random initial health values for equipment based on configured ranges."""
    ranges = config['health_ranges'][equipment_type]
    return {
        component: round(random.uniform(lo, hi), 3)
        for component, (lo, hi) in ranges.items()
    }

# Test
for eq_type in ['turbine', 'compressor', 'pump']:
    print(f"{eq_type}: {generate_initial_health(CONFIG, eq_type)}")

In [ ]:
# Create environmental model for Bonny Island using REAL weather data
print("\nSetting up Weather API for Bonny Island...")
print("Note: This uses real weather data. Set use_real_weather=False for synthetic data.")

# Create hybrid environment with real weather and synthetic fallback
fallback = EnvironmentalConditions(LocationType.TROPICAL)

bonny_island_env = create_hybrid_environment(
use_real_weather=True,
    api_provider="weatherapi",
    api_key = os.getenv('WEATHER_API_KEY'),
    location_name="Port Harcourt",  
    country="Nigeria",
    fallback_source=fallback,  
    cache_enabled=True
)
# This fetches hourly weather once, then uses cached data for 5-min interpolation
print(f"Pre-caching weather data for {CONFIG['duration_days']} days...")
bonny_island_env.preload_cache(
    start_date=CONFIG['start_date'],
    end_date=CONFIG['start_date'] + timedelta(days=CONFIG['duration_days']),
    interval_hours=1  
)

# Test environmental conditions
test_conditions = bonny_island_env.get_conditions(
    elapsed_hours=0, 
    timestamp=CONFIG['start_date']
)
print("\nBonny Island Environmental Conditions (July 1, 2025):")
print(f"  Temperature: {test_conditions['ambient_temp_C']:.1f}°C (REAL DATA)")
print(f"  Humidity: {test_conditions['humidity_percent']:.1f}%")
print(f"  Pressure: {test_conditions['pressure_kPa']:.1f} kPa")
# print(f"  Wind Speed: {test_conditions['wind_speed_m_s']:.1f} m/s")
print(f"  Corrosion factor: {test_conditions['corrosion_factor']:.2f}x (coastal)")
print(f"  Fouling factor: {test_conditions['fouling_factor']:.2f}x")

In [ ]:
conn = sqlite3.connect("weather_cache.db")

env_model = pd.read_sql_query(
    "SELECT * FROM weather_cache",
    conn
)

conn.close()

# Convert timestamps
env_model["timestamp"] = pd.to_datetime(env_model["timestamp"], utc=True)

# Select July 1, 2025 at 00:00
row = env_model.loc[env_model["timestamp"] == "2025-12-25T14:00:00Z"].iloc[0]

# print("\nBonny Island Environmental Conditions (July 1, 2025):")
print(f"  Temperature: {row['ambient_temp_C']:.1f}°C (REAL DATA)")
print(f"  Humidity: {row['humidity_percent']:.1f}%")
print(f"  Pressure: {row['pressure_kPa']:.1f} kPa")

In [ ]:
# Seed master data for all equipment using DataPipeline
print("SEEDING MASTER DATA FOR ALL EQUIPMENT")

try:
    # Initialize data pipeline
    db_url = os.getenv('POSTGRES_URL')
    pipeline = DataPipeline(db_url)
    pipeline.connect()
    
    print("\nDatabase connection successful\n")
    
    # Seed master data
    turbine_ids, compressor_ids, pump_ids = pipeline.seed_master_data(
        turbine_count=CONFIG['n_turbines'],
        compressor_count=CONFIG['n_compressors'],
        pump_count=CONFIG['n_pumps']
    )
    

    print("MASTER DATA SEEDING COMPLETE")
    print(f"\nTotal equipment created: {len(turbine_ids) + len(compressor_ids) + len(pump_ids)}")
    print(f"  - Gas Turbines: {len(turbine_ids)} (IDs: {turbine_ids[:5]}{'...' if len(turbine_ids) > 5 else ''})")
    print(f"  - Compressors: {len(compressor_ids)} (IDs: {compressor_ids[:5]}{'...' if len(compressor_ids) > 5 else ''})")
    print(f"  - Pumps: {len(pump_ids)} (IDs: {pump_ids[:6]}{'...' if len(pump_ids) > 6 else ''})")
    
    # Retrieve and display sample master data
    print("SAMPLE MASTER DATA FROM DATABASE")
    
    # Get turbine configs
    turbine_configs = pipeline.master_data.get_configs(turbine_ids[:3], 'turbine')
    print("\nGas Turbines (first 3):")
    df_turbines = pd.DataFrame(turbine_configs)
    display(df_turbines)
    
    # Get compressor configs
    compressor_configs = pipeline.master_data.get_configs(compressor_ids[:3], 'compressor')
    print("\nCompressors (first 3):")
    df_compressors = pd.DataFrame(compressor_configs)
    display(df_compressors)
    
    # Get pump configs
    pump_configs = pipeline.master_data.get_configs(pump_ids[:3], 'pump')
    print("\nPumps (first 3):")
    df_pumps = pd.DataFrame(pump_configs)
    display(df_pumps)
    
    # Close database connection
    pipeline.close()
    print("\nDatabase connection closed")
    
except Exception as e:
    print(f"\nError: {e}")
    print("Please ensure PostgreSQL is running and credentials are correct")
    raise

In [ ]:
print(f"\nSimulating {CONFIG['n_turbines']} Gas Turbines...")
turbine_telemetry = []
turbine_failures = []
turbine_maintenance = []

for i in range(CONFIG['n_turbines']):
    turbine_id = i + 1
    
    # Create turbine with initial health
    initial_health = generate_initial_health(CONFIG, 'turbine')
    print(f"GT-{turbine_id:03d} initial health: {initial_health}")
    
    turbine = GasTurbine(
        name=f"GT-{turbine_id:03d}",
        initial_health=initial_health,
        env_model=env_model,
        enable_environmental=True,
        enable_maintenance=True,  
        enable_thermal_transients=True,
        enable_enhanced_vibration=True,
        output_mode = OutputMode.DERIVED_FEATURES
    )
    
    # Simulate with maintenance/repair after failures
    for record in simulate_equipment(
        turbine, 
        turbine_id, 
        'turbine',
        CONFIG['duration_days'], 
        CONFIG['sample_interval_min'],
        start_time=CONFIG['start_date'],
        degradation_multiplier=CONFIG['degradation_multiplier'],
        include_equipment_type=True,
        maintenance_downtime_hours=24.0  # 24 hours downtime for repairs
    ):
        if record['type'] == 'failure':
            turbine_failures.append({
                'equipment_id': record['equipment_id'],
                'equipment_type': record['equipment_type'],
                'failure_time': record['failure_time'],
                'operating_hours_at_failure': record['operating_hours_at_failure'],
                'failure_mode_code': record['failure_mode_code'],
                'state': record['state']
            })
        elif record['type'] == 'maintenance_start':
            turbine_maintenance.append({
                'equipment_id': record['equipment_id'],
                'equipment_type': record['equipment_type'],
                'start_time': record['start_time'],
                'failure_code': record['failure_code'],
                'repaired_components': record['repaired_components'],
                'downtime_hours': record['downtime_hours']
            })
        elif record['type'] == 'telemetry':
            turbine_telemetry.append({
                'equipment_id': record['equipment_id'],
                'equipment_type': record['equipment_type'],
                'sample_time': record['sample_time'],
                'operating_hours': record['operating_hours'],
                'in_maintenance': record.get('in_maintenance', False),
                **record['state']
            })
    
    sample_count = len([r for r in turbine_telemetry if r['equipment_id'] == turbine_id])
    failure_count = len([f for f in turbine_failures if f['equipment_id'] == turbine_id])
    print(f"  GT-{turbine_id:03d}: {sample_count:,} samples, {failure_count} failures")

print(f"\nTotal turbine telemetry: {len(turbine_telemetry):,} records")
print(f"Total turbine failures: {len(turbine_failures)}")
print(f"Total turbine maintenance events: {len(turbine_maintenance)}")

In [ ]:
print(f"\nSimulating {CONFIG['n_compressors']} Compressors...")
compressor_telemetry = []
compressor_failures = []
compressor_maintenance = []

for i in range(CONFIG['n_compressors']):
    compressor_id = i + 1
    
    # Create compressor with initial health
    initial_health = generate_initial_health(CONFIG, 'compressor')
    print(f"COMP-{compressor_id:03d} initial health: {initial_health}")
    
    compressor = Compressor(
        name=f"COMP-{compressor_id:03d}",
        initial_health=initial_health,
        design_flow=random.uniform(1200, 1800),
        design_head=random.uniform(7000, 9000),
        env_model=env_model,
        enable_environmental=True,
        enable_maintenance=True,  
        enable_thermal_transients=True,
        enable_enhanced_vibration=True,
        output_mode = OutputMode.DERIVED_FEATURES
    )
    
    # Simulate with maintenance/repair after failures
    for record in simulate_equipment(
        compressor, 
        compressor_id, 
        'compressor',
        CONFIG['duration_days'], 
        CONFIG['sample_interval_min'],
        start_time=CONFIG['start_date'],
        degradation_multiplier=CONFIG['degradation_multiplier'],
        include_equipment_type=True,
        maintenance_downtime_hours=24.0  # 24 hours downtime for repairs
    ):
        if record['type'] == 'failure':
            compressor_failures.append({
                'equipment_id': record['equipment_id'],
                'equipment_type': record['equipment_type'],
                'failure_time': record['failure_time'],
                'operating_hours_at_failure': record['operating_hours_at_failure'],
                'failure_mode_code': record['failure_mode_code'],
                'state': record['state']
            })
        elif record['type'] == 'maintenance_start':
            compressor_maintenance.append({
                'equipment_id': record['equipment_id'],
                'equipment_type': record['equipment_type'],
                'start_time': record['start_time'],
                'failure_code': record['failure_code'],
                'repaired_components': record['repaired_components'],
                'downtime_hours': record['downtime_hours']
            })
        elif record['type'] == 'telemetry':
            compressor_telemetry.append({
                'equipment_id': record['equipment_id'],
                'equipment_type': record['equipment_type'],
                'sample_time': record['sample_time'],
                'operating_hours': record['operating_hours'],
                'in_maintenance': record.get('in_maintenance', False),
                **record['state']
            })
    
    sample_count = len([r for r in compressor_telemetry if r['equipment_id'] == compressor_id])
    failure_count = len([f for f in compressor_failures if f['equipment_id'] == compressor_id])
    print(f"  CC-{compressor_id:03d}: {sample_count:,} samples, {failure_count} failures")

print(f"\nTotal compressor telemetry: {len(compressor_telemetry):,} records")
print(f"Total compressor failures: {len(compressor_failures)}")
print(f"Total compressor maintenance events: {len(compressor_maintenance)}")

In [ ]:
print(f"\nSimulating {CONFIG['n_pumps']} Pumps...")
pump_telemetry = []
pump_failures = []
pump_maintenance = []

pump_services = [
    {'name': 'Crude Booster', 'design_flow': 200, 'design_head': 100, 'density': 850},
    {'name': 'Seawater Injection', 'design_flow': 300, 'design_head': 150, 'density': 1025},
    {'name': 'Process Water', 'design_flow': 100, 'design_head': 60, 'density': 1000},
    {'name': 'Methanol Pump', 'design_flow': 50, 'design_head': 80, 'density': 790},
    {'name': 'Fire Water', 'design_flow': 400, 'design_head': 120, 'density': 1000},
]

for i in range(CONFIG['n_pumps']):
    pump_id = i + 1
    service = pump_services[i % len(pump_services)]
    
    # Create pump with initial health
    initial_health = generate_initial_health(CONFIG, 'pump')
    
    pump = Pump(
        name=f"PUMP-{pump_id:03d}",
        initial_health=initial_health,
        design_flow=service['design_flow'] * random.uniform(0.9, 1.1),
        design_head=service['design_head'] * random.uniform(0.9, 1.1),
        design_speed=3000 + random.randint(-500, 500),
        fluid_density=service['density'],
        npsh_available=random.uniform(6, 12),
        env_model=env_model,
        enable_environmental=True,  
        enable_maintenance=True,  
        enable_thermal_transients=True,
        enable_enhanced_vibration=True
    )
    
    # Simulate with maintenance/repair after failures
    for record in simulate_equipment(
        pump, 
        pump_id, 
        'pump',
        CONFIG['duration_days'], 
        CONFIG['sample_interval_min'],
        start_time=CONFIG['start_date'],
        degradation_multiplier=CONFIG['degradation_multiplier'],
        include_equipment_type=True,
        maintenance_downtime_hours=12.0  # 12 hours downtime for pump repairs (simpler equipment)
    ):
        if record['type'] == 'failure':
            pump_failures.append({
                'equipment_id': record['equipment_id'],
                'equipment_type': record['equipment_type'],
                'failure_time': record['failure_time'],
                'operating_hours_at_failure': record['operating_hours_at_failure'],
                'failure_mode_code': record['failure_mode_code'],
                'state': record['state']
            })
        elif record['type'] == 'maintenance_start':
            pump_maintenance.append({
                'equipment_id': record['equipment_id'],
                'equipment_type': record['equipment_type'],
                'start_time': record['start_time'],
                'failure_code': record['failure_code'],
                'repaired_components': record['repaired_components'],
                'downtime_hours': record['downtime_hours']
            })
        elif record['type'] == 'telemetry':
            pump_telemetry.append({
                'equipment_id': record['equipment_id'],
                'equipment_type': record['equipment_type'],
                'sample_time': record['sample_time'],
                'operating_hours': record['operating_hours'],
                'in_maintenance': record.get('in_maintenance', False),
                **record['state']
            })
    
    if i % 5 == 4:  # Print every 5 pumps
        print(f"  CP-{pump_id-4:03d} to CP-{pump_id:03d} completed")

print(f"\nTotal pump telemetry: {len(pump_telemetry):,} records")
print(f"Total pump failures: {len(pump_failures)}")
print(f"Total pump maintenance events: {len(pump_maintenance)}")

  CP-001 to CP-005 completed
  CP-006 to CP-010 completed


In [ ]:
pump_failures

In [ ]:
db_url = os.getenv('POSTGRES_URL')
engine = create_engine(db_url)

In [ ]:
# Write turbine telemetry to database
df_turbine_telemetry = pd.DataFrame(turbine_telemetry)

if not df_turbine_telemetry.empty:
    df_db = df_turbine_telemetry.copy()
    
    # Extract derived features from JSON 'features' column into separate columns
    if 'features' in df_db.columns:
        features_parsed = df_db['features'].dropna().apply(
            lambda x: json.loads(x) if isinstance(x, str) else {}
        )
        features_df = pd.DataFrame(features_parsed.tolist(), index=features_parsed.index)
        for col in features_df.columns:
            df_db[col] = features_df[col]
    
    # Column mapping: DataFrame column name -> DB column name
    # Based on gas_turbine.py next_state() keys and 03_create_telemetry_tables.sql schema
    turbine_col_map = {
        'equipment_id': 'turbine_id',
        'sample_time': 'sample_time',
        'operating_hours': 'operating_hours',
        'speed': 'speed_rpm',
        'speed_target': 'speed_target_rpm',
        'exhaust_gas_temp': 'egt_celsius',
        'oil_temp': 'oil_temp_celsius',
        'fuel_flow': 'fuel_flow_kg_s',
        'compressor_discharge_temp': 'compressor_discharge_temp_celsius',
        'compressor_discharge_pressure': 'compressor_discharge_pressure_kpa',
        'efficiency': 'efficiency_fraction',
        'ambient_temp': 'ambient_temp_celsius',
        'ambient_pressure': 'ambient_pressure_kpa',
        'vibration_rms': 'vibration_rms_mm_s',
        'vibration_peak': 'vibration_peak_mm_s',
        'vibration_crest_factor': 'vibration_crest_factor',
        'vibration_kurtosis': 'vibration_kurtosis',
        'health_hgp': 'health_hgp',
        'health_blade': 'health_blade',
        'health_bearing': 'health_bearing',
        'health_fuel': 'health_fuel',
        'num_active_faults': 'num_active_faults',
        'total_faults_initiated': 'total_faults_initiated',
        'upset_active': 'upset_active',
        'upset_type': 'upset_type',
        'upset_severity': 'upset_severity',
        # Derived features (extracted from JSON above)
        'vibration_trend_7d': 'vibration_trend_7d',
        'temp_variation_24h': 'temp_variation_24h',
        'speed_stability': 'speed_stability',
        'efficiency_degradation_rate': 'efficiency_degradation_rate',
        'load_factor': 'load_factor',
    }
    
    # Select only columns present in DataFrame and rename to DB schema
    available = [c for c in turbine_col_map if c in df_db.columns]
    df_db = df_db[available].rename(columns=turbine_col_map)
    
    print(f"\nWriting {len(df_db):,} turbine telemetry records to database...")
    print(f"Columns ({len(df_db.columns)}): {list(df_db.columns)}")
    

    try:
        df_db.to_sql(
            name='gas_turbine_telemetry',
            schema='telemetry',
            con=engine,
            if_exists='append',
            index=False,
            chunksize=1000,
            method='multi'
        )
        print("Turbine telemetry written successfully")
    except Exception as e:
        print(f"Error writing turbine telemetry: {e}")

In [ ]:
# Write compressor telemetry to database
df_compressor_telemetry = pd.DataFrame(compressor_telemetry)

if not df_compressor_telemetry.empty:
    df_db = df_compressor_telemetry.copy()
    
    # Extract derived features from JSON 'features' column into separate columns
    if 'features' in df_db.columns:
        features_parsed = df_db['features'].dropna().apply(
            lambda x: json.loads(x) if isinstance(x, str) else {}
        )
        features_df = pd.DataFrame(features_parsed.tolist(), index=features_parsed.index)
        for col in features_df.columns:
            df_db[col] = features_df[col]
    
    # Column mapping: DataFrame column name -> DB column name
    # Based on compressor.py next_state() keys and 03_create_telemetry_tables.sql schema
    compressor_col_map = {
        'equipment_id': 'compressor_id',
        'sample_time': 'sample_time',
        'operating_hours': 'operating_hours',
        'speed': 'speed_rpm',
        'speed_target': 'speed_target_rpm',
        'flow': 'flow_m3h',
        'head': 'head_kj_kg',
        'efficiency': 'efficiency_fraction',
        'power': 'power_kw',
        'suction_pressure': 'suction_pressure_kpa',
        'suction_temp': 'suction_temp_celsius',
        'discharge_pressure': 'discharge_pressure_kpa',
        'discharge_temp': 'discharge_temp_celsius',
        'surge_margin': 'surge_margin_percent',
        'surge_alarm': 'surge_alarm',
        'orbit_amplitude': 'vibration_amplitude_mm',
        'sync_amplitude': 'sync_amplitude_mm',
        'shaft_x_displacement': 'shaft_x_displacement_mm',
        'shaft_y_displacement': 'shaft_y_displacement_mm',
        'bearing_temp_de': 'bearing_temp_de_celsius',
        'bearing_temp_nde': 'bearing_temp_nde_celsius',
        'thrust_bearing_temp': 'thrust_bearing_temp_celsius',
        'health_seal_primary': 'seal_health_primary',
        'health_seal_secondary': 'seal_health_secondary',
        'primary_seal_leakage': 'primary_seal_leakage_kg_s',
        'secondary_seal_leakage': 'secondary_seal_leakage_kg_s',
        'health_impeller': 'health_impeller',
        'health_bearing': 'health_bearing',
        'num_active_faults': 'num_active_faults',
        'total_faults_initiated': 'total_faults_initiated',
        'upset_active': 'upset_active',
        'upset_type': 'upset_type',
        'upset_severity': 'upset_severity',
        # Derived features (extracted from JSON above)
        'vibration_trend_7d': 'vibration_trend_7d',
        'temp_variation_24h': 'temp_variation_24h',
        'speed_stability': 'speed_stability',
        'efficiency_degradation_rate': 'efficiency_degradation_rate',
        'pressure_ratio': 'pressure_ratio',
        'load_factor': 'load_factor',
    }
    
    # Select only columns present in DataFrame and rename to DB schema
    available = [c for c in compressor_col_map if c in df_db.columns]
    df_db = df_db[available].rename(columns=compressor_col_map)
    
    print(f"\nWriting {len(df_db):,} compressor telemetry records to database...")
    print(f"Columns ({len(df_db.columns)}): {list(df_db.columns)}")
    
    try:
        df_db.to_sql(
            name='compressor_telemetry',
            schema='telemetry',
            con=engine,
            if_exists='append',
            index=False,
            chunksize=1000,
            method='multi'
        )
        print("Compressor telemetry written successfully")
    except Exception as e:
        print(f"Error writing compressor telemetry: {e}")

In [ ]:
# Write pump telemetry to database
df_pump_telemetry = pd.DataFrame(pump_telemetry)

if not df_pump_telemetry.empty:
    df_db = df_pump_telemetry.copy()
    
    # Extract derived features from JSON 'features' column into separate columns
    if 'features' in df_db.columns:
        features_parsed = df_db['features'].dropna().apply(
            lambda x: json.loads(x) if isinstance(x, str) else {}
        )
        if not features_parsed.empty:
            features_df = pd.DataFrame(features_parsed.tolist(), index=features_parsed.index)
            for col in features_df.columns:
                df_db[col] = features_df[col]
    
    # Column mapping: DataFrame column name -> DB column name
    # Based on pump.py next_state() keys and 03_create_telemetry_tables.sql schema
    pump_col_map = {
        'equipment_id': 'pump_id',
        'sample_time': 'sample_time',
        'operating_hours': 'operating_hours',
        'speed': 'speed_rpm',
        'speed_target': 'speed_target_rpm',
        'flow': 'flow_m3h',
        'head': 'head_m',
        'efficiency': 'efficiency_fraction',
        'power': 'power_kw',
        'suction_pressure': 'suction_pressure_kpa',
        'discharge_pressure': 'discharge_pressure_kpa',
        'fluid_temp': 'fluid_temp_celsius',
        'npsh_available': 'npsh_available_m',
        'npsh_required': 'npsh_required_m',
        'npsh_margin': 'cavitation_margin_m',
        'cavitation_severity': 'cavitation_severity',
        'vibration_rms': 'vibration_rms_mm_s',
        'vibration_peak': 'vibration_peak_mm_s',
        'bearing_temp_de': 'bearing_temp_de_celsius',
        'bearing_temp_nde': 'bearing_temp_nde_celsius',
        'motor_current': 'motor_current_amps',
        'motor_current_ratio': 'motor_current_ratio',
        'seal_leakage': 'seal_leakage_rate',
        'bep_deviation': 'bep_deviation_percent',
        'health_impeller': 'health_impeller',
        'health_seal': 'health_seal',
        'health_bearing_de': 'health_bearing_de',
        'health_bearing_nde': 'health_bearing_nde',
        # Derived features (extracted from JSON above)
        'vibration_trend_7d': 'vibration_trend_7d',
        'speed_stability': 'speed_stability',
        'efficiency_degradation_rate': 'efficiency_degradation_rate',
        'pressure_ratio': 'pressure_ratio',
        'load_factor': 'load_factor',
    }
    
    # Select only columns present in DataFrame and rename to DB schema
    available = [c for c in pump_col_map if c in df_db.columns]
    df_db = df_db[available].rename(columns=pump_col_map)
    
    print(f"\nWriting {len(df_db):,} pump telemetry records to database...")
    print(f"Columns ({len(df_db.columns)}): {list(df_db.columns)}")
    
    try:
        df_db.to_sql(
            name='pump_telemetry',
            schema='telemetry',
            con=engine,
            if_exists='append',
            index=False,
            chunksize=1000,
            method='multi'
        )
        print("Pump telemetry written successfully")
    except Exception as e:
        print(f"Error writing pump telemetry: {e}")

In [ ]:
# Write failure records to separate equipment-specific tables
df_turbine_failures = pd.DataFrame(turbine_failures)
df_compressor_failures = pd.DataFrame(compressor_failures)
df_pump_failures = pd.DataFrame(pump_failures)

print(f"Failure records by type:")
print(f"  Turbines:    {len(df_turbine_failures)}")
print(f"  Compressors: {len(df_compressor_failures)}")
print(f"  Pumps:       {len(df_pump_failures)}")

#  Turbine failures 
if not df_turbine_failures.empty:
    df_db = pd.DataFrame({
        'turbine_id': df_turbine_failures['equipment_id'],
        'failure_time': df_turbine_failures['failure_time'],
        'operating_hours_at_failure': df_turbine_failures['operating_hours_at_failure'],
        'failure_mode_code': df_turbine_failures['failure_mode_code'],
        'speed_rpm_at_failure': df_turbine_failures['state'].apply(lambda s: s.get('speed', 0)),
        'egt_celsius_at_failure': df_turbine_failures['state'].apply(lambda s: s.get('exhaust_gas_temp', 0)),
        'vibration_mm_s_at_failure': df_turbine_failures['state'].apply(lambda s: s.get('vibration_rms', 0)),
    })
    try:
        df_db.to_sql('gas_turbine_failures', schema='failure_events', con=engine,
                      if_exists='append', index=False, chunksize=500, method='multi')
        print(f"  Wrote {len(df_db)} turbine failure records")
    except Exception as e:
        print(f"  Error writing turbine failures: {e}")

#  Compressor failures 
if not df_compressor_failures.empty:
    df_db = pd.DataFrame({
        'compressor_id': df_compressor_failures['equipment_id'],
        'failure_time': df_compressor_failures['failure_time'],
        'operating_hours_at_failure': df_compressor_failures['operating_hours_at_failure'],
        'failure_mode_code': df_compressor_failures['failure_mode_code'],
        'speed_rpm_at_failure': df_compressor_failures['state'].apply(lambda s: s.get('speed', 0)),
        'surge_margin_at_failure': df_compressor_failures['state'].apply(lambda s: s.get('surge_margin', 0)),
        'vibration_amplitude_at_failure': df_compressor_failures['state'].apply(lambda s: s.get('orbit_amplitude', 0)),
    })
    try:
        df_db.to_sql('compressor_failures', schema='failure_events', con=engine,
                      if_exists='append', index=False, chunksize=500, method='multi')
        print(f"  Wrote {len(df_db)} compressor failure records")
    except Exception as e:
        print(f"  Error writing compressor failures: {e}")

#  Pump failures 
if not df_pump_failures.empty:
    df_db = pd.DataFrame({
        'pump_id': df_pump_failures['equipment_id'],
        'failure_time': df_pump_failures['failure_time'],
        'operating_hours_at_failure': df_pump_failures['operating_hours_at_failure'],
        'failure_mode_code': df_pump_failures['failure_mode_code'],
        'speed_rpm_at_failure': df_pump_failures['state'].apply(lambda s: s.get('speed', 0)),
        'vibration_mm_s_at_failure': df_pump_failures['state'].apply(lambda s: s.get('vibration_rms', 0)),
        'cavitation_margin_at_failure': df_pump_failures['state'].apply(lambda s: s.get('npsh_margin', 0)),
    })
    try:
        df_db.to_sql('pump_failures', schema='failure_events', con=engine,
                      if_exists='append', index=False, chunksize=500, method='multi')
        print(f"  Wrote {len(df_db)} pump failure records")
    except Exception as e:
        print(f"  Error writing pump failures: {e}")

print("\nFailure records write complete")

In [ ]:
with engine.connect() as conn:
    # Telemetry counts
    print("\nTelemetry Records:")
    for table, label in [
        ('telemetry.gas_turbine_telemetry', 'Turbines'),
        ('telemetry.compressor_telemetry', 'Compressors'),
        ('telemetry.pump_telemetry', 'Pumps'),
    ]:
        count = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
        print(f"  {label:15s} {count:>12,}")

    # Failure counts
    print("\nFailure Events:")
    for table, label in [
        ('failure_events.gas_turbine_failures', 'Turbines'),
        ('failure_events.compressor_failures', 'Compressors'),
        ('failure_events.pump_failures', 'Pumps'),
    ]:
        count = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
        print(f"  {label:15s} {count:>12,}")

    # Time range sanity check
    print("\nTime Ranges:")
    for table, label in [
        ('telemetry.gas_turbine_telemetry', 'Turbines'),
        ('telemetry.compressor_telemetry', 'Compressors'),
        ('telemetry.pump_telemetry', 'Pumps'),
    ]:
        row = conn.execute(text(
            f"SELECT MIN(sample_time), MAX(sample_time) FROM {table}"
        )).fetchone()
        if row and row[0]:
            print(f"  {label:15s} {row[0]} → {row[1]}")

# Close database connection
engine.dispose()
print("\nDatabase connection closed")
print("SIMULATION & INGESTION COMPLETE")